# Visualizing differences in expression with pathways

In [1]:
import pandas as pd
import numpy as np
import statsmodels.stats
import gseapy as gp

import cptac
import cptac.utils as ut

In [2]:
cc = cptac.Ccrcc()

In [3]:
lists = ut.get_hgnc_protein_lists()

In [5]:
complex_name = "L ribosomal proteins"
complex_members = lists[complex_name]

Select columns from the proteomics dataframe for proteins that are part of the complex.

In [6]:
prot = cc.get_proteomics()
complex_prot = prot[(prot.columns.get_level_values(0) & complex_members).drop_duplicates()]
print(complex_prot.columns.size)
complex_prot.head()

194


Name,RPL10,RPL10A,RPL11,RPL12,RPL13A,RPL14,RPL15,RPL17,RPL18A,RPL19,...,RPL5,RPL6,RPL7,RPL7A,RPL7L1,RPL8,RPL9,RPLP0,RPLP1,RPLP2
Database_ID,NP_001290553.1,NP_009035.3,NP_000966.2,NP_000967.1,NP_036555.1,NP_001030168.1,NP_001240308.1,NP_000976.1,NP_000971.1,NP_000972.1,...,NP_000960.2,NP_000961.2,NP_000962.2,NP_000963.1,NP_940888.3,NP_000964.1,NP_000652.2,NP_000993.1,NP_000994.1,NP_000995.1
Patient_ID,,,,,,,,,,,,,,,,,,,,,
C3L-00004,0.264960,0.309660,0.183279,0.326624,0.652286,0.004129,0.408317,0.289828,0.447878,0.338549,...,0.414334,0.501916,0.571614,0.355377,0.230248,0.307144,0.300112,0.185480,0.241690,0.254670
C3L-00010,0.085429,0.076285,0.113017,0.159180,0.452463,0.356429,0.513856,0.170330,0.250976,0.314508,...,0.261360,0.557743,0.535490,0.382357,0.388502,0.228932,-0.022451,-0.089075,0.247563,0.007501
C3L-00011,0.419416,0.337406,0.321088,0.319416,0.646663,0.474013,0.510343,0.508749,0.482465,0.694802,...,0.398840,0.601783,0.519981,0.482813,0.455874,0.524241,0.354969,0.333386,0.350022,0.319232
C3L-00026,0.042099,-0.077893,-0.141792,-0.135201,0.390498,0.159345,0.242294,0.054415,-0.119543,-0.098429,...,0.362816,0.249324,0.225179,0.181407,-0.035459,0.172209,-0.187755,-0.071052,-0.294384,0.202132
C3L-00079,0.332185,0.306407,0.297075,0.225482,0.395265,0.170959,0.436913,0.206110,0.372101,0.506096,...,0.289113,0.186728,0.430170,0.306386,0.592443,0.234061,0.251575,0.222530,0.243519,0.204548


Find proteins with a significant difference in expression between tumor and normal cells.

In [8]:
ids = []
diffs = []
p_vals = []

for col in complex_prot.columns:
    
    # Separate out tumor and normal
    data = complex_prot[col]
    tumor = data[~data.index.str.endswith(".N")]
    normal = data[data.index.str.endswith(".N")]
    
    # Permutation test
    diff, p_val, null_dist = ut.permutation_test_means(tumor, normal, num_permutations=10000)

    # Record results
    ids.append(col)
    diffs.append(diff)
    p_vals.append(p_val)

In [9]:
results = pd.DataFrame({"change": diffs, "P_val": p_vals}, index=pd.MultiIndex.from_tuples(ids))

Simplify change in abundance: Increase is 1, decrease is -1, no change stays 0.

In [10]:
results.insert(1, "simplified_change", results["change"])
results["simplified_change"][results["change"] > 0] = 1
results["simplified_change"][results["change"] < 0] = -1
results["simplified_change"][results["change"] == 0] = 0

Multiple testing correction.

In [11]:
reject_null, adj_p, alpha_sidak, alpha_bonf = statsmodels.stats.multitest.multipletests(
    pvals=results["P_val"],
    alpha=0.05,
    method="fdr_bh")

results = results.assign(adj_p=adj_p, reject_null=reject_null)

Use GSEApy to find pathways enriched for genes in these lists.

In [12]:
sig_res = results[results["reject_null"]]
sig_genes = results.index.get_level_values(0).tolist()

In [13]:
sig_res

,,change,simplified_change,P_val,adj_p,reject_null
RPL10,NP_001290553.1,0.123576,1.0,0.0000,0.000000,True
RPL10A,NP_009035.3,0.115490,1.0,0.0000,0.000000,True
RPL11,NP_000966.2,0.149255,1.0,0.0000,0.000000,True
RPL12,NP_000967.1,0.186145,1.0,0.0000,0.000000,True
RPL13A,NP_036555.1,0.798172,1.0,0.0000,0.000000,True
RPL14,NP_001030168.1,0.294289,1.0,0.0000,0.000000,True
RPL15,NP_001240308.1,0.569601,1.0,0.0000,0.000000,True
RPL17,NP_000976.1,0.229385,1.0,0.0000,0.000000,True
RPL18A,NP_000971.1,0.364704,1.0,0.0000,0.000000,True
RPL19,NP_000972.1,0.689389,1.0,0.0000,0.000000,True


In [16]:
increase_enriched_reactome = gp.enrichr(
    gene_list=sig_genes,
    description="increased_in_tumor",
    gene_sets="Reactome_2016",
    outdir="enrichr/reactome")

In [17]:
pi = increase_enriched_reactome.res2d
pi.insert(1, "pathway_id", pi["Term"].str.rsplit(" ", n=1, expand=True)[1])

In [18]:
pi.head()

,Term,pathway_id,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Gene_set
0,Viral mRNA Translation Homo sapiens R-HSA-192823,R-HSA-192823,43/84,1.130017e-105,1.728927e-102,0,0,227.513228,54978.389816,RPL4;RPL5;RPL30;RPL3;RPL10;RPL32;RPL31;RPL12;R...,Reactome_2016
1,Peptide chain elongation Homo sapiens R-HSA-15...,R-HSA-156902,43/84,1.130017e-105,8.644633e-103,0,0,227.513228,54978.389816,RPL4;RPL5;RPL30;RPL3;RPL10;RPL32;RPL31;RPL12;R...,Reactome_2016
2,Selenocysteine synthesis Homo sapiens R-HSA-24...,R-HSA-2408557,43/87,9.045461e-105,4.613185e-102,0,0,219.667944,52625.667337,RPL4;RPL5;RPL30;RPL3;RPL10;RPL32;RPL31;RPL12;R...,Reactome_2016
3,Eukaryotic Translation Termination Homo sapien...,R-HSA-72764,43/87,9.045461e-105,3.459889e-102,0,0,219.667944,52625.667337,RPL4;RPL5;RPL30;RPL3;RPL10;RPL32;RPL31;RPL12;R...,Reactome_2016
4,Eukaryotic Translation Elongation Homo sapiens...,R-HSA-156842,43/89,3.422784e-104,1.047372e-101,0,0,214.731586,51157.308400,RPL4;RPL5;RPL30;RPL3;RPL10;RPL32;RPL31;RPL12;R...,Reactome_2016


In [19]:
simp = sig_res.copy(deep=True)[["simplified_change"]]
simp.index = simp.index.droplevel(1)
simp.index.name = "protein"
simp = simp.reset_index()
simp = simp.drop_duplicates()
simp = simp.set_index("protein")
simp.head()

,simplified_change
protein,
RPL10,1.0
RPL10A,1.0
RPL11,1.0
RPL12,1.0
RPL13A,1.0


In [20]:
for pathway in pi.pathway_id[:5]:
    ut.reactome_pathway_overlay(
        df=simp,
        pathway=pathway,
        open_browser=True)

In [13]:
# Get complex proteins
# Use permutation tests to find significantly different
# Correct with statsmodels
# Use GSEApy to find pathways where those are overrepresented
# Visualize in Reactome

In [30]:
test.iloc[:, 1].notna().any()

True